In [ ]:
from sentinelhub import DataCollection

for collection in DataCollection.get_available_collections():
    print(collection)

# A. Import + config

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from datetime import datetime, timedelta

from sentinelhub import (
    CRS,
    BBox,
    DataCollection,
    MimeType,
    MosaickingOrder,
    SentinelHubRequest,
    SHConfig,
    bbox_to_dimensions,
)

# =========================
# CONFIG CDSE
# =========================
config = SHConfig()

config.sh_client_id = "sh-aba2f8e9-da97-47f3-8cae-a5a1fbeede2d"
config.sh_client_secret = "DPBslOq7z6pumggnCpQAQBDKfIE3fhQ9"

config.sh_token_url = "https://identity.dataspace.copernicus.eu/auth/realms/CDSE/protocol/openid-connect/token"
config.sh_base_url = "https://sh.dataspace.copernicus.eu"


# B. Create CDSE collections

In [ ]:
# Sentinel-2 L2A for CDSE
s2l2a_cdse = DataCollection.SENTINEL2_L2A.define_from(
    "S2L2A_CDSE",
    service_url=config.sh_base_url
)

# Sentinel-5P for CDSE
s5p_cdse = DataCollection.SENTINEL5P.define_from(
    "S5P_CDSE",
    service_url=config.sh_base_url
)

# C. Helper functions

In [ ]:
def make_bbox(lat, lon, delta, crs=CRS.WGS84):
    return BBox([lon - delta, lat - delta, lon + delta, lat + delta], crs=crs)


def safe_stats(arr, prefix):
    arr = np.array(arr, dtype=np.float32)
    arr = arr[~np.isnan(arr)]
    
    if arr.size == 0:
        return {
            f"{prefix}_mean": np.nan,
            f"{prefix}_std": np.nan,
            f"{prefix}_min": np.nan,
            f"{prefix}_max": np.nan,
            f"{prefix}_median": np.nan,
        }
    
    return {
        f"{prefix}_mean": float(np.nanmean(arr)),
        f"{prefix}_std": float(np.nanstd(arr)),
        f"{prefix}_min": float(np.nanmin(arr)),
        f"{prefix}_max": float(np.nanmax(arr)),
        f"{prefix}_median": float(np.nanmedian(arr)),
    }

4) Sentinel-5P Feature Extractor
D. Sentinel-5P feature extractor

In [ ]:
# from datetime import datetime, timedelta
# import numpy as np

# def get_s5p_single_band_features(lat, lon, date_str, band_name, delta=0.05, resolution=2000, min_qa=75):
#     bbox = make_bbox(lat, lon, delta)
#     size = bbox_to_dimensions(bbox, resolution=resolution)

#     start_dt = datetime.strptime(date_str, "%Y-%m-%d")
#     end_dt = start_dt + timedelta(days=1)

#     time_from = start_dt.strftime("%Y-%m-%dT00:00:00Z")
#     time_to   = end_dt.strftime("%Y-%m-%dT00:00:00Z")

#     evalscript = f"""
#     //VERSION=3
#     function setup() {{
#         return {{
#             input: [{{
#                 bands: ["{band_name}", "dataMask"]
#             }}],
#             output: {{
#                 bands: 2,
#                 sampleType: "FLOAT32"
#             }}
#         }};
#     }}

#     function evaluatePixel(sample) {{
#         return [sample.{band_name}, sample.dataMask];
#     }}
#     """

#     request = SentinelHubRequest(
#         evalscript=evalscript,
#         input_data=[
#             SentinelHubRequest.input_data(
#                 data_collection=s5p_cdse,
#                 time_interval=(time_from, time_to),
#                 mosaicking_order=MosaickingOrder.MOST_RECENT,
#                 other_args={
#                     "dataFilter": {
#                         "timeliness": "RPRO"   # or "OFFL"
#                     },
#                     "processing": {
#                         "minQa": min_qa
#                     }
#                 }
#             )
#         ],
#         responses=[
#             SentinelHubRequest.output_response("default", MimeType.TIFF)
#         ],
#         bbox=bbox,
#         size=size,
#         config=config
#     )

#     try:
#         img = request.get_data()[0]
#     except Exception as e:
#         print(f"S5P {band_name} error on {date_str}: {e}")
#         return {
#             f"{band_name.lower()}_valid_pixels": 0,
#             f"{band_name.lower()}_valid_ratio": 0.0,
#             **safe_stats([], band_name.lower())
#         }

#     band = img[..., 0]
#     mask = img[..., 1]

#     band_masked = np.where(mask == 1, band, np.nan)
#     valid_pixels = int(np.sum(~np.isnan(band_masked)))
#     total_pixels = int(band_masked.size)
#     valid_ratio = valid_pixels / total_pixels if total_pixels > 0 else 0.0

#     features = {
#         f"{band_name.lower()}_valid_pixels": valid_pixels,
#         f"{band_name.lower()}_valid_ratio": valid_ratio
#     }
#     features.update(safe_stats(band_masked, band_name.lower()))
#     return features

In [ ]:
# def get_s5p_single_band_features(lat, lon, date_str, band_name, delta=0.1, resolution=5000):
#     """
#     Fetch features for one Sentinel-5P band for one day.
#     band_name examples: 'NO2', 'CO', 'SO2', 'AER_AI_340_380'
#     """
#     bbox = make_bbox(lat, lon, delta)
#     size = bbox_to_dimensions(bbox, resolution=resolution)

#     print(f"{date_str} | delta={delta} | bbox={bbox} | size={size}")
    
#     evalscript = f"""
#     //VERSION=3
#     function setup() {{
#         return {{
#             input: [{{
#                 bands: ["{band_name}", "dataMask"]
#             }}],
#             output: {{
#                 bands: 2,
#                 sampleType: "FLOAT32"
#             }}
#         }};
#     }}

#     function evaluatePixel(sample) {{
#         return [sample.{band_name}, sample.dataMask];
#     }}
#     """

#     request = SentinelHubRequest(
#         evalscript=evalscript,
#         input_data=[
#             SentinelHubRequest.input_data(
#                 data_collection=s5p_cdse,
#                 time_interval=(date_str, date_str),
#                 mosaicking_order=MosaickingOrder.MOST_RECENT
#             )
#         ],
#         responses=[
#             SentinelHubRequest.output_response("default", MimeType.TIFF)
#         ],
#         bbox=bbox,
#         size=size,
#         config=config
#     )

#     try:
#         img = request.get_data()[0]
#     except Exception as e:
#         print(f"S5P {band_name} error on {date_str}: {e}")
#         return {
#             f"{band_name.lower()}_valid_pixels": np.nan,
#             **safe_stats([], band_name.lower())
#         }

#     band = img[..., 0]
#     mask = img[..., 1]

#     band_masked = np.where(mask == 1, band, np.nan)

#     features = {
#         f"{band_name.lower()}_valid_pixels": int(np.sum(~np.isnan(band_masked)))
#     }
#     features.update(safe_stats(band_masked, band_name.lower()))
#     return features

In [ ]:
# from datetime import datetime, timedelta
# import numpy as np

# def get_s5p_single_band_features(lat, lon, date_str, band_name, delta=0.05, resolution=2000, min_qa=50):
#     bbox = make_bbox(lat, lon, delta)
#     size = bbox_to_dimensions(bbox, resolution=resolution)

#     start_dt = datetime.strptime(date_str, "%Y-%m-%d")
#     end_dt = start_dt + timedelta(days=1)

#     time_from = start_dt.strftime("%Y-%m-%dT00:00:00Z")
#     time_to   = end_dt.strftime("%Y-%m-%dT00:00:00Z")

#     evalscript = f"""
#     //VERSION=3
#     function setup() {{
#         return {{
#             input: [{{
#                 bands: ["{band_name}", "dataMask"]
#             }}],
#             output: {{
#                 bands: 2,
#                 sampleType: "FLOAT32"
#             }}
#         }};
#     }}

#     function evaluatePixel(sample) {{
#         return [sample.{band_name}, sample.dataMask];
#     }}
#     """

#     request = SentinelHubRequest(
#         evalscript=evalscript,
#         input_data=[
#             SentinelHubRequest.input_data(
#                 data_collection=s5p_cdse,
#                 time_interval=(time_from, time_to),
#                 mosaicking_order=MosaickingOrder.MOST_RECENT,
#                 other_args={
#                     "processing": {
#                         "minQa": min_qa
#                     }
#                 }
#             )
#         ],
#         responses=[
#             SentinelHubRequest.output_response("default", MimeType.TIFF)
#         ],
#         bbox=bbox,
#         size=size,
#         config=config
#     )

#     try:
#         img = request.get_data()[0]
#         print(f"[DEBUG] {date_str} | band={band_name} | img shape={img.shape}")
#     except Exception as e:
#         print(f"[ERROR] S5P {band_name} on {date_str}: {e}")
#         return {
#             f"{band_name.lower()}_valid_pixels": 0,
#             f"{band_name.lower()}_valid_ratio": 0.0,
#             f"{band_name.lower()}_mean": np.nan,
#             f"{band_name.lower()}_std": np.nan,
#             f"{band_name.lower()}_min": np.nan,
#             f"{band_name.lower()}_max": np.nan,
#             f"{band_name.lower()}_median": np.nan,
#         }

#     band = img[..., 0]
#     mask = img[..., 1]

#     print(f"[DEBUG] mask unique for {date_str}: ", np.unique(mask, return_counts=True))

#     band_masked = np.where(mask == 1, band, np.nan)

#     valid_pixels = int(np.sum(~np.isnan(band_masked)))
#     total_pixels = int(band_masked.size)
#     valid_ratio = valid_pixels / total_pixels if total_pixels > 0 else 0.0

#     print(f"[DEBUG] valid_pixels={valid_pixels}, total_pixels={total_pixels}, valid_ratio={valid_ratio:.4f}")

#     return {
#         f"{band_name.lower()}_valid_pixels": valid_pixels,
#         f"{band_name.lower()}_valid_ratio": valid_ratio,
#         f"{band_name.lower()}_mean": np.nanmean(band_masked) if valid_pixels > 0 else np.nan,
#         f"{band_name.lower()}_std": np.nanstd(band_masked) if valid_pixels > 0 else np.nan,
#         f"{band_name.lower()}_min": np.nanmin(band_masked) if valid_pixels > 0 else np.nan,
#         f"{band_name.lower()}_max": np.nanmax(band_masked) if valid_pixels > 0 else np.nan,
#         f"{band_name.lower()}_median": np.nanmedian(band_masked) if valid_pixels > 0 else np.nan,
#     }

In [ ]:
import pandas as pd
import numpy as np
from tqdm import tqdm
from datetime import datetime, timedelta

def get_s5p_single_band_features(
    lat,
    lon,
    date_str,
    band_name,
    delta=0.05,
    resolution=2000,
    min_qa=None,
    timeliness=None,
    debug=False,
):
    """
    Fetch Sentinel-5P features for one band for one day.
    band_name: 'NO2', 'SO2', 'O3'
    """
    bbox = make_bbox(lat, lon, delta)
    size = bbox_to_dimensions(bbox, resolution=resolution)

    start_dt = datetime.strptime(date_str, "%Y-%m-%d")
    end_dt = start_dt + timedelta(days=1)

    time_from = start_dt.strftime("%Y-%m-%dT00:00:00Z")
    time_to = end_dt.strftime("%Y-%m-%dT00:00:00Z")

    evalscript = f"""
    //VERSION=3
    function setup() {{
        return {{
            input: [{{
                bands: ["{band_name}", "dataMask"]
            }}],
            output: {{
                bands: 2,
                sampleType: "FLOAT32"
            }}
        }};
    }}

    function evaluatePixel(sample) {{
        return [sample.{band_name}, sample.dataMask];
    }}
    """

    other_args = {}
    if timeliness is not None:
        other_args["dataFilter"] = {"timeliness": timeliness}
    if min_qa is not None:
        other_args["processing"] = {"minQa": min_qa}

    request = SentinelHubRequest(
        evalscript=evalscript,
        input_data=[
            SentinelHubRequest.input_data(
                data_collection=s5p_cdse,
                time_interval=(time_from, time_to),
                mosaicking_order=MosaickingOrder.MOST_RECENT,
                other_args=other_args if other_args else None
            )
        ],
        responses=[
            SentinelHubRequest.output_response("default", MimeType.TIFF)
        ],
        bbox=bbox,
        size=size,
        config=config
    )

    prefix = band_name.lower()

    try:
        img = request.get_data()[0]
        if debug:
            print(f"[DEBUG] {date_str} | {band_name} | img shape = {img.shape}")
    except Exception as e:
        print(f"[ERROR] {band_name} | {date_str} | {e}")
        return {
            f"{prefix}_valid_pixels": 0,
            f"{prefix}_valid_ratio": 0.0,
            **safe_stats([], prefix)
        }

    band = img[..., 0]
    mask = img[..., 1]

    if debug:
        print(f"[DEBUG] {date_str} | {band_name} | mask unique = {np.unique(mask, return_counts=True)}")

    band_masked = np.where(mask == 1, band, np.nan)

    valid_pixels = int(np.sum(~np.isnan(band_masked)))
    total_pixels = int(band_masked.size)
    valid_ratio = valid_pixels / total_pixels if total_pixels > 0 else 0.0

    features = {
        f"{prefix}_valid_pixels": valid_pixels,
        f"{prefix}_valid_ratio": valid_ratio,
    }
    features.update(safe_stats(band_masked, prefix))
    return features


In [ ]:

def fetch_daily_s5p_features(
    lat,
    lon,
    unique_dates,
    product_configs,
    delta=0.05,
    resolution=2000,
    debug=False
):
    """
    Fetch multiple Sentinel-5P gas products by day.
    Returns a dict: {"NO2": df_no2, "SO2": df_so2, "O3": df_o3}
    """
    all_sat_daily = {}

    for gas, cfg in product_configs.items():
        print(f"\n===== Fetching {gas} =====")
        sat_rows = []

        for date_str in tqdm(unique_dates, desc=gas):
            feats = get_s5p_single_band_features(
                lat=lat,
                lon=lon,
                date_str=date_str,
                band_name=gas,
                delta=delta,
                resolution=resolution,
                min_qa=cfg.get("min_qa"),
                timeliness=cfg.get("timeliness"),
                debug=debug
            )

            row = {"sat_date": date_str}
            row.update(feats)
            sat_rows.append(row)

        sat_df = pd.DataFrame(sat_rows)

        # Rename columns to mark them as daily features
        prefix = gas.lower()
        sat_df = sat_df.rename(columns={
            f"{prefix}_valid_pixels": f"{prefix}_daily_valid_pixels",
            f"{prefix}_valid_ratio": f"{prefix}_daily_valid_ratio",
            f"{prefix}_mean": f"{prefix}_daily_mean",
            f"{prefix}_std": f"{prefix}_daily_std",
            f"{prefix}_min": f"{prefix}_daily_min",
            f"{prefix}_max": f"{prefix}_daily_max",
            f"{prefix}_median": f"{prefix}_daily_median",
        })

        all_sat_daily[gas] = sat_df

    return all_sat_daily

In [ ]:
# import pandas as pd
# from tqdm import tqdm

# # =========================
# # 1) Read the OpenAQ file
# # =========================
# openaq = pd.read_csv("data/raw/DataSample/03. openaq_01032025_31032025_cmt8.csv")
# openaq["datetimeUtc"] = pd.to_datetime(openaq["datetimeUtc"], utc=True)

# # If the file contains multiple locations, filter the target location
# # The uploaded file currently has one CMT8 location, but keep this guard for safety:
# target_location = "CMT8"
# openaq = openaq[openaq["location_name"] == target_location].copy()

# # =========================
# # 2) Convert long format to wide format
# #    Each timestamp becomes one row
# # =========================
# openaq_wide = (
#     openaq.pivot_table(
#         index=["datetimeUtc", "location_name", "latitude", "longitude"],
#         columns="parameter",
#         values="value",
#         aggfunc="mean"
#     )
#     .reset_index()
#     .sort_values("datetimeUtc")
#     .reset_index(drop=True)
# )

# openaq_wide.columns.name = None

# # Rename columns for easier use
# openaq_wide = openaq_wide.rename(columns={
#     "pm25": "pm25",
#     "relativehumidity": "rh",
#     "temperature": "temp"
# })

# # =========================
# # 3) Extract one coordinate from the OpenAQ file
# # =========================
# lat = float(openaq_wide["latitude"].iloc[0])
# lon = float(openaq_wide["longitude"].iloc[0])
# location_name = openaq_wide["location_name"].iloc[0]

# print(f"Using ONE location only: {location_name} ({lat}, {lon})")
# print("OpenAQ time range:",
#       openaq_wide["datetimeUtc"].min(),
#       "->",
#       openaq_wide["datetimeUtc"].max())

# # =========================
# # 4) Create the list of unique dates
# #    so satellite data is requested once per day
# # =========================
# openaq_wide["sat_date"] = openaq_wide["datetimeUtc"].dt.strftime("%Y-%m-%d")
# unique_dates = sorted(openaq_wide["sat_date"].unique())

# print(f"Hourly rows in OpenAQ: {len(openaq_wide)}")
# print(f"Unique UTC dates to fetch from satellite: {len(unique_dates)}")

# # =========================
# # 5) Request satellite data for the selected location,
# #    but only once per day
# # =========================
# sat_rows = []

# for date_str in tqdm(unique_dates):
#     try:
#         features = get_s5p_single_band_features(
#             lat=lat,
#             lon=lon,
#             date_str=date_str,
#             band_name="O3"
#         )

#         if features is None:
#             features = {}

#         row = {"sat_date": date_str}
#         row.update(features)
#         sat_rows.append(row)

#     except Exception as e:
#         print(f"[WARN] {date_str}: {e}")
#         sat_rows.append({"sat_date": date_str})

# sat_df = pd.DataFrame(sat_rows)

# # =========================
# # 6) Merge satellite features
# #    back into the hourly OpenAQ table
# # =========================
# merged = (
#     openaq_wide
#     .merge(sat_df, on="sat_date", how="left")
#     .sort_values("datetimeUtc")
#     .reset_index(drop=True)
# )

# # =========================
# # 7) Optionally keep only rows with PM2.5
# # =========================
# merged = merged[merged["pm25"].notna()].copy()

# # =========================
# # 8) Save results
# # =========================
# merged.to_csv("data/processed/s5p/openaq_cmt8_hourly_0103202531032025_with_s5p_O3.csv", index=False)

# print("Saved: openaq_cmt8_hourly_0101202531012025_with_s5p_O3.csv")
# print(merged.head())
# print(merged.columns.tolist())



In [ ]:
# =========================================================
# 1) CONFIGURATION
# =========================================================
OPENAQ_CSV = "data/raw/DataSample/03. openaq_01032025_31032025_cmt8.csv"
OUTPUT_CSV = "data/processed/s5p/openaq_cmt8_hourly_merged_NO2_SO2_O3.csv"
TARGET_LOCATION = "CMT8"

DELTA = 0.05
RESOLUTION = 2000

PRODUCT_CONFIGS = {
    "NO2": {
        "min_qa": 50,
        "timeliness": None
    },
    "SO2": {
        "min_qa": 50,
        "timeliness": None
    },
    "O3": {
        "min_qa": None,
        "timeliness": None
    }
}


# =========================================================
# 2) READ OPENAQ
# =========================================================
openaq = pd.read_csv(OPENAQ_CSV)
openaq["datetimeUtc"] = pd.to_datetime(openaq["datetimeUtc"], utc=True)

# Filter the target station
openaq = openaq[openaq["location_name"] == TARGET_LOCATION].copy()

print("OpenAQ raw shape:", openaq.shape)
print("Time range:", openaq["datetimeUtc"].min(), "->", openaq["datetimeUtc"].max())
print("\nParameter counts:")
print(openaq["parameter"].value_counts())


# =========================================================
# 3) CONVERT LONG TO WIDE
# =========================================================
openaq_wide = (
    openaq.pivot_table(
        index=["datetimeUtc", "location_name", "latitude", "longitude"],
        columns="parameter",
        values="value",
        aggfunc="mean"
    )
    .reset_index()
    .sort_values("datetimeUtc")
    .reset_index(drop=True)
)

openaq_wide.columns.name = None

openaq_wide = openaq_wide.rename(columns={
    "pm25": "pm25",
    "relativehumidity": "rh",
    "temperature": "temp"
})

lat = float(openaq_wide["latitude"].iloc[0])
lon = float(openaq_wide["longitude"].iloc[0])

print(f"\nUsing location: {TARGET_LOCATION} ({lat}, {lon})")

# Create sat_date to merge daily satellite data into hourly observations
openaq_wide["sat_date"] = openaq_wide["datetimeUtc"].dt.strftime("%Y-%m-%d")
unique_dates = sorted(openaq_wide["sat_date"].unique())

print("Hourly rows:", len(openaq_wide))
print("Unique daily satellite dates:", len(unique_dates))
print("Date range:", unique_dates[0], "->", unique_dates[-1])


# =========================================================
# 4) FETCH NO2 + SO2 + O3 TOGETHER
# =========================================================
all_sat_daily = fetch_daily_s5p_features(
    lat=lat,
    lon=lon,
    unique_dates=unique_dates,
    product_configs=PRODUCT_CONFIGS,
    delta=DELTA,
    resolution=RESOLUTION,
    debug=False
)


# =========================================================
# 5) MERGE EVERYTHING INTO ONE FILE
# =========================================================
merged = openaq_wide.copy()

for gas in ["NO2", "SO2", "O3"]:
    merged = merged.merge(all_sat_daily[gas], on="sat_date", how="left")

# Keep rows with pm25 if needed
merged = merged[merged["pm25"].notna()].copy()

# Reorder columns
base_cols = [
    "datetimeUtc", "sat_date", "location_name", "latitude", "longitude",
    "pm25", "rh", "temp"
]

sat_cols = [c for c in merged.columns if c not in base_cols]
merged = merged[base_cols + sat_cols]


# =========================================================
# 6) QUICK CHECK
# =========================================================
print("\nMerged shape:", merged.shape)
print("\nMissing per column:")
print(merged.isna().sum())

print("\nSample rows:")
print(merged.head())


# =========================================================
# 7) EXPORT ONE FILE
# =========================================================
merged.to_csv(OUTPUT_CSV, index=False)
print(f"\nSaved file: {OUTPUT_CSV}")

In [ ]:
daily_diag = pd.DataFrame({"sat_date": unique_dates})

for gas in ["NO2", "SO2", "O3"]:
    daily_diag = daily_diag.merge(all_sat_daily[gas], on="sat_date", how="left")

daily_diag.to_csv("data/processed/s5p/daily_s5p_coverage_NO2_SO2_O3.csv", index=False)
print("Saved file: data/processed/s5p/daily_s5p_coverage_NO2_SO2_O3.csv")

# Daily coverage diagnostics

In [ ]:
# # Configure parameters
# lat, lon = 10.7769, 106.7009  # HCM_1 station coordinates
# date_str = "2023-01-01"  # Specific date
# band_name = "NO2"  # Or "CO", "SO2", "AER_AI_340_380"
# features = get_s5p_single_band_features(lat, lon, date_str, band_name)
# print(features)

In [ ]:
# # Read station information from CSV
# stations = pd.read_csv("data/raw/stations/hcm_stations.csv")

# # Time range to fetch
# start_date = "2023-01-01"
# end_date = "2023-01-31"

# # Create a dictionary to store data
# all_features = {}

# # Iterate through stations
# for _, station in stations.iterrows():
#     lat = station['lat']
#     lon = station['lon']
#     station_name = station['station']

#     print(f"Fetching data for station {station_name} ({lat}, {lon}) from {start_date} to {end_date}")
    
#     # Iterate through days in the month
#     date_range = pd.date_range(start=start_date, end=end_date, freq='D')
    
#     station_features = {}
#     for date in date_range:
#         date_str = date.strftime('%Y-%m-%d')
#         features = get_s5p_single_band_features(lat, lon, date_str, band_name="NO2")
#         station_features[date_str] = features
    
#     # Store data for the station
#     all_features[station_name] = station_features

# # Inspect fetched data
# print(all_features)

In [ ]:
import pandas as pd

# Convert the all_features dictionary to a DataFrame
df_features = []

# Iterate through each station and date to create DataFrame rows
for station_name, station_data in all_features.items():
    for date_str, features in station_data.items():
        # Create one dict for each data row
        row = {'station': station_name, 'date': date_str}
        row.update(features)  # Update features for that row
        df_features.append(row)

# Convert the list of rows to a DataFrame
df_features_df = pd.DataFrame(df_features)

# Save the DataFrame to CSV
output_file = "data/processed/s5p/s5p_features_01_2023.csv"
df_features_df.to_csv(output_file, index=False)

print(f"Data exported to {output_file}")

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Filter data to get AOD and NO2
df = pd.read_csv("data/processed/modis/mcd19a2_feature_engineered.csv")
aod_values = df['mcd19a2_aod_055']  # AOD from MODIS
no2_values = features['no2_mean']  # NO2 from Sentinel-5P

# Create a DataFrame for visualization
visual_data = pd.DataFrame({
    'AOD': aod_values,
    'NO2': no2_values
})

# Create a scatter plot between AOD and NO2
plt.figure(figsize=(10, 6))
sns.scatterplot(data=visual_data, x='AOD', y='NO2')

plt.title('Relationship Between AOD and NO2', fontsize=16)
plt.xlabel('AOD (MODIS)', fontsize=12)
plt.ylabel('NO2 (Sentinel-5P)', fontsize=12)
plt.show()

In [ ]:
# Create a heatmap to inspect the correlation between AOD and NO2
correlation_matrix = visual_data.corr()

plt.figure(figsize=(6, 4))
sns.heatmap(correlation_matrix, annot=True, cmap="coolwarm", fmt=".2f", cbar=True)
plt.title('Correlation Between AOD and NO2', fontsize=16)
plt.show()

5) Sentinel-2 Feature Extractor
E. Sentinel-2 feature extractor

In [ ]:
# def get_s2_features(lat, lon, date_str, delta=0.01, resolution=10):
#     """
#     Fetch Sentinel-2 L2A features for one day.
#     date_str: 'YYYY-MM-DD'
#     """
#     bbox = make_bbox(lat, lon, delta)
#     size = bbox_to_dimensions(bbox, resolution=resolution)

#     evalscript = """
#     //VERSION=3
#     function setup() {
#         return {
#             input: [{
#                 bands: ["B02", "B03", "B04", "B08", "B11", "dataMask"],
#                 units: "REFLECTANCE"
#             }],
#             output: {
#                 bands: 6,
#                 sampleType: "FLOAT32"
#             }
#         };
#     }

#     function evaluatePixel(sample) {
#         return [
#             sample.B02,
#             sample.B03,
#             sample.B04,
#             sample.B08,
#             sample.B11,
#             sample.dataMask
#         ];
#     }
#     """

#     request = SentinelHubRequest(
#         evalscript=evalscript,
#         input_data=[
#             SentinelHubRequest.input_data(
#                 data_collection=s2l2a_cdse,
#                 time_interval=(date_str, date_str),
#                 mosaicking_order=MosaickingOrder.LEAST_CC
#             )
#         ],
#         responses=[
#             SentinelHubRequest.output_response("default", MimeType.TIFF)
#         ],
#         bbox=bbox,
#         size=size,
#         config=config
#     )

#     try:
#         img = request.get_data()[0]
#     except Exception as e:
#         print(f"S2 error on {date_str}: {e}")
#         return {
#             "date": date_str,
#             "s2_valid_pixels": np.nan,
#             **safe_stats([], "ndvi"),
#             **safe_stats([], "ndbi"),
#             **safe_stats([], "ndwi"),
#         }

#     b2 = img[..., 0]
#     b3 = img[..., 1]
#     b4 = img[..., 2]
#     b8 = img[..., 3]
#     b11 = img[..., 4]
#     mask = img[..., 5]

#     b2 = np.where(mask == 1, b2, np.nan)
#     b3 = np.where(mask == 1, b3, np.nan)
#     b4 = np.where(mask == 1, b4, np.nan)
#     b8 = np.where(mask == 1, b8, np.nan)
#     b11 = np.where(mask == 1, b11, np.nan)

#     # Indices
#     ndvi = (b8 - b4) / (b8 + b4 + 1e-6)
#     ndbi = (b11 - b8) / (b11 + b8 + 1e-6)
#     ndwi = (b3 - b8) / (b3 + b8 + 1e-6)

#     features = {
#         "date": date_str,
#         "s2_valid_pixels": int(np.sum(~np.isnan(ndvi)))
#     }
#     features.update(safe_stats(ndvi, "ndvi"))
#     features.update(safe_stats(ndbi, "ndbi"))
#     features.update(safe_stats(ndwi, "ndwi"))

#     return features

6) Build Dataset by Date
F. Create the Date List

In [ ]:
# def generate_date_list(start_date, end_date):
#     dates = pd.date_range(start_date, end_date, freq="D")
#     return [d.strftime("%Y-%m-%d") for d in dates]

G. Build full feature table

In [ ]:
# def build_satellite_feature_dataset(lat, lon, start_date, end_date):
#     date_list = generate_date_list(start_date, end_date)

#     rows = []
#     for d in date_list:
#         print(f"Processing {d} ...")
        
#         s5p_feat = get_s5p_features(lat, lon, d)
#         s2_feat = get_s2_features(lat, lon, d)

#         row = {"date": d, "lat": lat, "lon": lon}
#         row.update({k: v for k, v in s5p_feat.items() if k != "date"})
#         row.update({k: v for k, v in s2_feat.items() if k != "date"})

#         dt = pd.to_datetime(d)
#         row["day_of_week"] = dt.dayofweek
#         row["month"] = dt.month
#         row["day_of_year"] = dt.dayofyear

#         rows.append(row)

#     return pd.DataFrame(rows)

7) Example Satellite Dataset Run

In [ ]:
# lat = 10.78533
# lon = 106.67029

# df_sat = build_satellite_feature_dataset(
#     lat=lat,
#     lon=lon,
#     start_date="2024-01-01",
#     end_date="2024-01-10"
# )

# df_sat.head()

H. Load PM2.5 CSV

In [ ]:
df_pm25 = pd.read_csv("data_test.csv")
df_pm25["date"] = pd.to_datetime(df_pm25["date"]).dt.strftime("%Y-%m-%d")

I. Merge

In [ ]:
df_full = df_sat.merge(df_pm25, on="date", how="left")
df_full.head()

9) Create Lag Features for Forecasting

If the target is PM2.5 on day t, use features from previous days.

J. Lag features

In [ ]:
lag_cols = [
    "no2_mean", "no2_std", "co_mean", "so2_mean", "aer_ai_mean",
    "ndvi_mean", "ndbi_mean", "ndwi_mean"
]

for col in lag_cols:
    if col in df_full.columns:
        df_full[f"{col}_lag1"] = df_full[col].shift(1)
        df_full[f"{col}_lag2"] = df_full[col].shift(2)
        df_full[f"{col}_lag3"] = df_full[col].shift(3)

# Current label
df_model = df_full.dropna(subset=["PM2.5"]).copy()

10) Prepare Model Training
K. Select Features

In [ ]:
feature_cols = [
    c for c in df_model.columns
    if c not in ["date", "pm25"]
]

X = df_model[feature_cols]
y = df_model["PM2.5"]

print(X.shape, y.shape)

11) Train a Baseline Model
L. Random Forest baseline

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, shuffle=False
)

model = RandomForestRegressor(
    n_estimators=200,
    random_state=42
)

model.fit(X_train, y_train)

y_pred = model.predict(X_test)

print("MAE :", mean_absolute_error(y_test, y_pred))
print("RMSE:", mean_squared_error(y_test, y_pred, squared=False))
print("R2  :", r2_score(y_test, y_pred))

In [ ]:
print(df_pm25.columns.tolist())
print(df_pm25.head())

In [ ]:
print("df_sat shape:", df_sat.shape)
print("df_pm25 shape:", df_pm25.shape)
print("df_full shape:", df_full.shape)
print("df_model shape:", df_model.shape)

print(df_full[["date", "PM2.5"]].head(10))
print(df_full["PM2.5"].isna().sum())

12) Save dataset

In [ ]:
df_sat.to_csv("satellite_features.csv", index=False)
df_full.to_csv("pm25_full_dataset.csv", index=False)
df_model.to_csv("pm25_model_dataset.csv", index=False)

13) If Weather Data Is Available

Merge it by date:

In [ ]:
df_weather = pd.read_csv("weather_daily.csv")
df_weather["date"] = pd.to_datetime(df_weather["date"]).dt.strftime("%Y-%m-%d")

df_full = df_full.merge(df_weather, on="date", how="left")